In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin


In [8]:
url = "https://download.bls.gov/pub/time.series/pr/"

# BLS requests that you include an email address so they can contact you
# if there are issues with your usage. Replace the email below with yours.
bls_email = "sbatlouni@gmail.com"
headers = {
    "User-Agent": f"requests_library ({bls_email})",
}
response = requests.get(url, headers=headers)

In [9]:
response.raise_for_status()

# The page is an IIS directory listing: each file line is
# "<date> <time> <AM/PM> <size> <a href=...>name</a>" inside a <pre> block.
soup = BeautifulSoup(response.text, "html.parser")
rows = []
for a in soup.find("pre").find_all("a"):
    meta = a.previous_sibling
    if not isinstance(meta, str) or not meta.strip():
        continue  # skips "[To Parent Directory]"
    date, time, ampm, size = meta.split()
    rows.append({
        "file_name": a.get_text(),
        "last_modified": pd.to_datetime(f"{date} {time} {ampm}", format="%m/%d/%Y %I:%M %p"),
        "size_bytes": int(size),
        "url": urljoin(url, a["href"]),
    })

bls_metadata = pd.DataFrame(rows)
bls_metadata

,file_name,last_modified,size_bytes,url
0,pr.class,2026-09-03 08:30:00,102,https://download.bls.gov/pub/time.series/pr/pr...
1,pr.contacts,2022-09-13 16:52:00,562,https://download.bls.gov/pub/time.series/pr/pr...
2,pr.data.0.Current,2026-09-03 08:30:00,1615931,https://download.bls.gov/pub/time.series/pr/pr...
3,pr.data.1.AllData,2026-09-03 08:30:00,3239525,https://download.bls.gov/pub/time.series/pr/pr...
4,pr.duration,2026-09-03 08:30:00,176,https://download.bls.gov/pub/time.series/pr/pr...
5,pr.footnote,2026-09-03 08:30:00,40,https://download.bls.gov/pub/time.series/pr/pr...
6,pr.measure,2026-09-03 08:30:00,745,https://download.bls.gov/pub/time.series/pr/pr...
7,pr.period,1994-01-07 15:53:00,146,https://download.bls.gov/pub/time.series/pr/pr...
8,pr.seasonal,2011-11-18 16:05:00,79,https://download.bls.gov/pub/time.series/pr/pr...
9,pr.sector,2026-09-03 08:30:00,263,https://download.bls.gov/pub/time.series/pr/pr...


In [ ]:
# Read the table once into a {file_name: last_modified} dictionary.
existing_rows = spark.table("rearc.raw.bls_metadata").select("file_name", "last_modified").collect()
existing_files = dict(existing_rows)

files_to_ingest = []
for file_name, last_modified in zip(bls_metadata["file_name"], bls_metadata["last_modified"]):
    if file_name not in existing_files:
        print(f"{file_name}: new")
        files_to_ingest.append(file_name)
    elif last_modified > existing_files[file_name]:
        print(f"{file_name}: updated ({existing_files[file_name]} -> {last_modified})")
        files_to_ingest.append(file_name)
    else:
        print(f"{file_name}: unchanged")

In [ ]:
import os
import time

volume_dir = "/Volumes/rearc/raw/volume/bls/pr"
os.makedirs(volume_dir, exist_ok=True)

# Download each new or updated file into the volume, overwriting any older copy.
downloaded_files = []
failed_files = []
for row in bls_metadata[bls_metadata["file_name"].isin(files_to_ingest)].itertuples():
    target_path = os.path.join(volume_dir, row.file_name)
    try:
        with requests.get(row.url, headers=headers, stream=True, timeout=60) as file_response:
            file_response.raise_for_status()
            with open(target_path, "wb") as f:
                for chunk in file_response.iter_content(chunk_size=1024 * 1024):
                    f.write(chunk)
        print(f"{row.file_name}: downloaded to {target_path}")
        downloaded_files.append(row.file_name)
    except Exception as e:
        print(f"{row.file_name}: failed ({e})")
        failed_files.append(row.file_name)
    time.sleep(1)  # be polite to the BLS server

In [ ]:
# Record only the files that downloaded, so failed files are picked up again on the next run.
if downloaded_files:
    spark.createDataFrame(
        bls_metadata[bls_metadata["file_name"].isin(downloaded_files)],
        schema=spark.table("rearc.raw.bls_metadata").schema,
    ).createOrReplaceTempView("bls_metadata_updates")

    spark.sql("""
        MERGE INTO rearc.raw.bls_metadata AS t
        USING bls_metadata_updates AS s
        ON t.file_name = s.file_name
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Recorded {len(downloaded_files)} file(s) in rearc.raw.bls_metadata")
else:
    print("Nothing new to record")

if failed_files:
    raise RuntimeError(f"Failed to download: {failed_files}")